## wd

In [ ]:
### Mounting Google Drive :
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pwd

In [ ]:
working_dir = "/content/drive/MyDrive/_PUCP_s/Cursos/1INF03_ADD/datos"
%cd $working_dir

In [ ]:
!pwd

In [ ]:
import pandas as pd
import numpy as np

# Modelos No Supervisados - Métodos para datos Mixtos

La clave es que Gower proporciona la distancia para datos mixtos; de manera que el algoritmo de clustering seleccionado deba poder trabajar con la matriz de distancia de estos datos mixtos.

In [ ]:
!pip install gower
!pip install kmodes scikit-learn-extra hdbscan

In [ ]:
x_scaled2 = pd.read_csv("x_scaled2_bin.csv")
x_scaled2.head(5)

In [ ]:
cat_cols = list(x_scaled2.select_dtypes(['object', 'string', 'category']).columns)
num_cols = list(x_scaled2.select_dtypes(['float64', 'int64']).columns)

In [ ]:
X = x_scaled2[num_cols + cat_cols].copy()

for col in cat_cols:
    X[col] = X[col].astype(str)

In [ ]:
import gower

D = gower.gower_matrix(X)
D

In [ ]:
print(type(D))
print(D.shape)

# a) K-Prototypes

Revisar notebook anterior

Tener en cuenta que K-Prototypes no utiliza directamente D. Necesita la matriz original de variables mixtas.

# b) Gower + Jerárquico

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

D_condensed = squareform(D, checks=False)
D_condensed

In [ ]:
Z = linkage(D_condensed, method="average")
Z

In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram

plt.figure(figsize=(10, 5))
dendrogram(Z)
plt.title("Dendrograma: Gower + Clustering Jerárquico")
plt.xlabel("Observaciones")
plt.ylabel("Distancia Gower")
plt.show()

* D → matriz cuadrada de distancias n × n.

* D_condensed → vector de distancias en formato condensado.

Ej de D_condensada: [0.2, 0.7, 0.5, 0.6, 0.4, 0.3]

Se interpreta como:
A-B = 0.2
A-C = 0.7
A-D = 0.5
B-C = 0.6
B-D = 0.4
C-D = 0.3

* Z → matriz de linkage generada a partir de las distancias.

Ej de Z, con method "average":
[[0.   1.   0.2  2. ]
 [2.   3.   0.3  2. ]
 [4.   5.   0.55 4. ]]


In [ ]:
# Número de clusters seleccionado
k = 4

clusters_hier = fcluster(Z, t=k, criterion="maxclust")

print(np.unique(clusters_hier, return_counts=True))


In [ ]:
# GRÁFICO - GOWER + JERÁRQUICO

import matplotlib.pyplot as plt
from sklearn.manifold import MDS

D_float = np.asarray(D, dtype=np.float64)

mds = MDS(
    n_components=2,
    dissimilarity="precomputed",
    random_state=42
)

coords = mds.fit_transform(D_float)

plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    coords[:, 0],
    coords[:, 1],
    c=clusters_hier,
    cmap="tab10",
    s=30,
    alpha=0.7
)

plt.colorbar(scatter, label="Cluster")
plt.title("Gower + Clustering Jerárquico")
plt.xlabel("Dimensión MDS 1")
plt.ylabel("Dimensión MDS 2")
plt.grid(alpha=0.2)

plt.show()


Tener en cuenta que:

Al reducir tanta información a solamente dos dimensiones, MDS puede crear separaciones visuales que no representan necesariamente clusters estadísticamente definidos.

Eso significa que, aunque visualmente parecen dos grupos, con el criterio de del modelo, no existe suficiente evidencia para considerarlos dos clusters dentro de los puntos de color rosado por ejemplo.

Interpretación: "La representación bidimensional mediante MDS muestra cierta estructura visual en los datos; sin embargo, el modelo (Jerárquico), utilizando las distancias Gower originales, no identificó dichas regiones lo suficientemente diferenciadas bajo los parámetros utilizados."

# c) Gower + PAM/K-Medoids

In [ ]:
from sklearn_extra.cluster import KMedoids

k = 3

pam = KMedoids(
    n_clusters=k,
    metric="precomputed",
    random_state=42
)

clusters_pam = pam.fit_predict(D)

print(np.unique(clusters_pam, return_counts=True))


In [ ]:
# GRÁFICO - GOWER + PAM

import matplotlib.pyplot as plt
from sklearn.manifold import MDS

D_float = np.asarray(D, dtype=np.float64)

mds = MDS(
    n_components=2,
    dissimilarity="precomputed",
    random_state=42
)

coords = mds.fit_transform(D_float)

plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    coords[:, 0],
    coords[:, 1],
    c=clusters_pam,
    cmap="tab10",
    s=30,
    alpha=0.7
)

plt.colorbar(scatter, label="Cluster")
plt.title("Gower + PAM (K-Medoids)")
plt.xlabel("Dimensión MDS 1")
plt.ylabel("Dimensión MDS 2")
plt.grid(alpha=0.2)

plt.show()


# d) Gower + DBSCAN

En DBSCAN, la etiqueda de -1 significa cluster considerado outlier o ruido.

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(
    eps=0.3,
    min_samples=5,
    metric="precomputed"
)

clusters_dbscan = dbscan.fit_predict(D)

print(np.unique(clusters_dbscan, return_counts=True))


In [ ]:
n_outliers = np.sum(clusters_dbscan == -1)

print("Número de outliers:", n_outliers)


## ¿Cómo elegir eps para DBSCAN?

Podemos mirar la distribución de distancias de cada observación a sus vecinos.

In [ ]:
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

k = 5

neigh = NearestNeighbors(
    n_neighbors=k,
    metric="precomputed"
)

neigh.fit(D)

distances, indices = neigh.kneighbors(D)

k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(10, 5))

plt.plot(k_distances)

plt.xlabel("Observaciones ordenadas")
plt.ylabel(f"Distancia al {k}° vecino")
plt.title("Curva k-distance para seleccionar eps")

plt.grid()
plt.show()


Si buscamos aproximadamente el "codo" de la curva, este valor puede servir como eps.

In [ ]:
dbscan = DBSCAN(
    eps=0.097,
    min_samples=50,
    metric="precomputed"
)

clusters_dbscan = dbscan.fit_predict(D)


In [ ]:
# GRÁFICO - GOWER + DBSCAN

import matplotlib.pyplot as plt
from sklearn.manifold import MDS

D_float = np.asarray(D, dtype=np.float64)

mds = MDS(
    n_components=2,
    dissimilarity="precomputed",
    random_state=42
)

coords = mds.fit_transform(D_float)

plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    coords[:, 0],
    coords[:, 1],
    c=clusters_dbscan,
    cmap="tab10",
    s=30,
    alpha=0.7
)

plt.colorbar(scatter, label="Cluster")
plt.title("Gower + DBSCAN")
plt.xlabel("Dimensión MDS 1")
plt.ylabel("Dimensión MDS 2")
plt.grid(alpha=0.2)

plt.show()


In [ ]:
# GRÁFICO TÍPICO DE DBSCAN

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import MDS

D_float = np.asarray(D, dtype=np.float64)

# Reducimos las distancias Gower a 2 dimensiones
mds = MDS(
    n_components=2,
    dissimilarity="precomputed",
    random_state=42
)

coords = mds.fit_transform(D_float)


# Separamos clusters y ruido
mask_outliers = clusters_dbscan == -1
mask_clusters = clusters_dbscan != -1

plt.figure(figsize=(11, 8))


# Clusters
plt.scatter(
    coords[mask_clusters, 0],
    coords[mask_clusters, 1],
    c=clusters_dbscan[mask_clusters],
    cmap="tab10",
    s=35,
    alpha=0.70,
    edgecolors="none"
)


# Outliers / ruido
plt.scatter(
    coords[mask_outliers, 0],
    coords[mask_outliers, 1],
    c="black",
    marker="x",
    s=45,
    linewidths=1.2,
    label="Ruido / Outliers"
)

plt.title("DBSCAN aplicado sobre distancia Gower", fontsize=15)
plt.xlabel("MDS 1")
plt.ylabel("MDS 2")

plt.grid(alpha=0.2)
plt.legend()

plt.show()


# e) Gower + HDBSCAN

HDBSCAN es muy interesante porque puede encontrar clusters con diferentes densidades.

In [ ]:
import hdbscan

D = np.asarray(D, dtype=np.float64)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=50,
    metric="precomputed"
)

clusters_hdbscan = clusterer.fit_predict(D)

print(np.unique(clusters_hdbscan, return_counts=True))


In [ ]:
n_outliers_hdbscan = np.sum(clusters_hdbscan == -1)

print("Outliers:", n_outliers_hdbscan)


In [ ]:
clusters_validos = set(clusters_hdbscan)

clusters_validos.discard(-1)

print("Número de clusters:", len(clusters_validos))


In [ ]:
# ============================================================
# GRÁFICO - GOWER + HDBSCAN
# ============================================================

import matplotlib.pyplot as plt
from sklearn.manifold import MDS

D_float = np.asarray(D, dtype=np.float64)

mds = MDS(
    n_components=2,
    dissimilarity="precomputed",
    random_state=42
)

coords = mds.fit_transform(D_float)

plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    coords[:, 0],
    coords[:, 1],
    c=clusters_hdbscan,
    cmap="tab10",
    s=30,
    alpha=0.7
)

plt.colorbar(scatter, label="Cluster")
plt.title("Gower + HDBSCAN")
plt.xlabel("Dimensión MDS 1")
plt.ylabel("Dimensión MDS 2")
plt.grid(alpha=0.2)

plt.show()


In [ ]:
plt.figure(figsize=(11, 8))

plt.scatter(
    coords[:, 0],
    coords[:, 1],
    c=clusters_hdbscan,
    cmap="tab10",
    s=18,
    alpha=0.55,
    edgecolors="none"
)

plt.title("Gower + HDBSCAN", fontsize=15)
plt.xlabel("MDS 1")
plt.ylabel("MDS 2")
plt.grid(alpha=0.15)

plt.show()


# Comparando los resultados

In [ ]:
resultados = pd.DataFrame(index=x_scaled2.index)

resultados["Jerarquico"] = clusters_hier
resultados["PAM"] = clusters_pam
resultados["DBSCAN"] = clusters_dbscan
resultados["HDBSCAN"] = clusters_hdbscan
#resultados["KPrototypes"] = clusters_kproto

resultados.head()


In [ ]:
for metodo in resultados.columns:

    print("----------------------")
    print(metodo)
    print("----------------------")

    print(resultados[metodo].value_counts().sort_index())


# Evaluación de los clusters con Silhouette

In [ ]:
from sklearn.metrics import silhouette_score

def evaluar_silhouette(D, labels, nombre):

    # Eliminamos outliers (-1)
    mask = labels != -1

    labels_validos = labels[mask]

    if len(np.unique(labels_validos)) < 2:
        print(nombre, ": no se puede calcular")
        return np.nan

    score = silhouette_score(
        D[np.ix_(mask, mask)],
        labels_validos,
        metric="precomputed"
    )

    print(nombre, ":", round(score, 4))

    return score


In [ ]:
sil_hier = evaluar_silhouette(
    D,
    clusters_hier,
    "Jerárquico"
)

sil_pam = evaluar_silhouette(
    D,
    clusters_pam,
    "PAM"
)

sil_dbscan = evaluar_silhouette(
    D,
    clusters_dbscan,
    "DBSCAN"
)


sil_hdbscan = evaluar_silhouette(
    D,
    clusters_hdbscan,
    "HDBSCAN"
)


In [ ]:
comparacion = pd.DataFrame({
    "Método": [
        "Jerárquico",
        "PAM",
        "DBSCAN",
        "HDBSCAN"
    ],
    "Silhouette": [
        sil_hier,
        sil_pam,
        sil_dbscan,
        sil_hdbscan
    ]
})

comparacion.sort_values(
    "Silhouette",
    ascending=False
)


"Aunque K-Prototypes utiliza una función de disimilitud propia para la construcción de los clusters, la evaluación comparativa de las soluciones se puede realizar mediante el coeficiente de silhouette calculado sobre la matriz de distancias Gower, con el propósito de emplear un criterio de evaluación común para todos los métodos de clustering aplicados a los datos mixtos."